# 01 — Datos y procesamiento del Asistente de Delivery Económico

**Autor:** Ruben Vigil · **Propósito:** ejecutar y explicar el proceso académico.

| Tareas | Qué muestra este cuaderno | Estado |
|---|---|---|
| T05–T06 | Elección de fuente y adquisición verificable | Ejecutable |
| T07 | Auditoría y definición del objetivo | Ejecutable |
| T08 | Preparación con Pandas | Ejecutable |
| T09 | EDA y visualizaciones | Ejecutable |
| T10–T12 | Dask, PySpark y comparación | Pendiente |

Los resultados salen de los scripts y manifiestos reales. Este cuaderno no entrena
modelos ni acredita precisión en El Salvador. T01–T04 prepararon el repositorio,
el entorno, la arquitectura y el dominio; su implementación se explica en README.

## Cómo ejecutarlo

**Local:** desde la raíz, ejecutar `uv sync --locked --extra notebooks` y abrir
este `.ipynb` en VS Code seleccionando el intérprete `.venv/bin/python`.
Ejecutar las celdas en orden.

**Colab:** subir este notebook desde **Archivo → Subir notebook**. Antes, generar
las fuentes actuales en la terminal local:

```bash
uv run --locked --extra notebooks python ml/data/package_notebook_source.py
```

Cuando la primera celda solicite un archivo, seleccionar
`data/reports/presentation/delivery_source.zip`. El ZIP contiene código y documentación;
no incluye `.env`, credenciales ni datos. Así se puede mostrar también T08 antes
de publicar esta rama. Regenerarlo después de cambiar el código.

La primera ejecución descarga la fuente oficial (ZIP de aproximadamente 39 MB)
y prepara el CSV en el entorno de ejecución. No solicita credenciales AWS ni monta
Drive. La instalación del proyecto usa su propio entorno con `uv`; los extras
`notebooks` permiten validar este mismo cuaderno localmente.

Colab utiliza una máquina temporal: guardar el notebook no conserva por sí solo
los datasets de esa máquina. Compartir el cuaderno puede compartir sus salidas;
aquí se muestran agregados y hashes, sin filas de pedidos.
[Referencia oficial de Colab](https://research.google.com/colaboratory/faq.html).

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
import zipfile

# Local: buscar el proyecto desde el directorio del notebook o sus padres.
PROJECT_DIR = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "ml/data/prepare_meituan.py").is_file()
     and (path / "pyproject.toml").is_file()),
    None,
)
if PROJECT_DIR is None:
    try:
        from google.colab import files
    except ImportError as error:
        raise RuntimeError("Abrí este notebook dentro del repositorio o en Colab.") from error
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Seleccioná únicamente delivery_source.zip")
    archive_path = Path(next(iter(uploaded)))
    PROJECT_DIR = Path(tempfile.mkdtemp(prefix="delivery-colab-"))
    with zipfile.ZipFile(archive_path) as archive:
        source_manifest = json.loads(archive.read("SOURCE_MANIFEST.json"))
        for name, expected_hash in source_manifest["files"].items():
            target = (PROJECT_DIR / name).resolve()
            if PROJECT_DIR not in target.parents:
                raise ValueError("Ruta inválida en el paquete de fuentes")
            content = archive.read(name)
            if hashlib.sha256(content).hexdigest() != expected_hash:
                raise ValueError(f"Hash de fuente inválido: {name}")
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(content)
    print("HEAD de referencia:", source_manifest["git_head"])
    print("Incluye cambios locales:", source_manifest["working_tree_changes"])
    print("Hashes del código verificados; el HEAD solo identifica la base.")
print("Proyecto:", PROJECT_DIR)

In [ ]:
# Aislar las dependencias del proyecto del kernel de Colab.
PROJECT_ENV = os.environ.copy()
PROJECT_ENV.pop("VIRTUAL_ENV", None)
uv_path = shutil.which("uv")
if uv_path:
    UV = [uv_path]
else:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "uv==0.12.23"], check=True
    )
    UV = [sys.executable, "-m", "uv"]
subprocess.run(UV + ["sync", "--locked", "--extra", "notebooks"], cwd=PROJECT_DIR, env=PROJECT_ENV, check=True)

def run_project(*arguments):
    result = subprocess.run(
        UV + ["run", "--locked", "--extra", "notebooks", *arguments],
        cwd=PROJECT_DIR, env=PROJECT_ENV, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    result.check_returncode()
    return result

def read_json(relative_path):
    return json.loads((PROJECT_DIR / relative_path).read_text(encoding="utf-8"))

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

from html import escape
from IPython.display import HTML, display

def show_table(headers, rows):
    head = "".join(f"<th>{escape(str(value))}</th>" for value in headers)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in row) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

import runpy
REVISION = runpy.run_path(str(PROJECT_DIR / "ml/data/acquire_meituan.py"))["REVISION"]
RAW_DIR = Path("data/raw/meituan") / REVISION
AUDIT_PATH = Path("data/reports/meituan") / REVISION / "t07_audit.json"
PROCESSED_DIR = Path("data/processed/meituan") / REVISION / "t08-v1"

## T05 — Por qué elegimos estos datos

**Objetivo:** contar con eventos reales para investigar duración de entrega.

Se compararon fuentes en `docs/DATA_ML.md`. Ruben seleccionó **Meituan INFORMS TSL
Research Challenge** porque publica datos y eventos temporales verificables.
La fuente original corresponde a China: permite estudiar el proceso de ciencia
de datos, pero no demuestra tiempos válidos para El Salvador o para cada vehículo.

El CSV registra **intentos de asignación/entrega (waybills)**, no necesariamente
pedidos diferentes. Por eso contar filas no basta para definir ejemplos de ML.
La licencia incluye CC BY-NC y condiciones adicionales del editor; se conserva
su texto junto al original. Este cuaderno no publica registros ni sube archivos a S3.

- [Repositorio oficial y documentación](https://github.com/meituan/Meituan-INFORMS-TSL-Research-Challenge).
- [Licencia de la revisión elegida](https://github.com/meituan/Meituan-INFORMS-TSL-Research-Challenge/blob/1f9b4288cee5a78d1e5da007fc306bbaa662fc6d/License.txt).

**Para explicar:** accesibilidad, licencia, calidad y representatividad son comprobaciones diferentes.

## T06 — Adquirir y conservar la fuente

**Objetivo:** obtener siempre la misma revisión sin modificar el original.

El script descarga el ZIP y la licencia, verifica tamaño y SHA-256, extrae solo
el CSV previsto y registra un manifiesto. Si la copia existe, comprueba su
integridad sin sobrescribirla. Un hash identifica bytes; no prueba calidad ni
exactitud predictiva.

In [ ]:
_ = run_project("python", "ml/data/acquire_meituan.py")
acquisition = read_json(RAW_DIR / "manifest.json")
show_table(["Archivo", "Bytes", "SHA-256"], [
    [name, info["bytes"], info["sha256"]]
    for name, info in acquisition["inventory"]["files"].items()
])
print("Revisión:", acquisition["revision"])
print("Filas originales:", acquisition["inventory"]["rows"])
print("Columnas originales:", len(acquisition["inventory"]["columns"]))
print("Licencia:", acquisition["license"])

## T07 — Auditar antes de preparar o entrenar

**Objetivo:** decidir qué observaciones sirven y qué queremos predecir.

Objetivo observado: `time_taken_min = (arrive_time - platform_order_time) / 60`.
Incluye desde creación hasta llegada; no representa únicamente conducción.
Aunque no haya celdas vacías, los ceros de eventos pueden indicar una entrega
no completada. Los reintentos tampoco son pedidos nuevos.

Se propone una fila por pedido inmediato, con datos de cotización estables,
una única entrega positiva, aceptación y eventos cronológicos válidos. Las
exclusiones se cuentan por primera causa, sin sumar dos veces el mismo pedido.

In [ ]:
_ = run_project("python", "ml/data/audit_meituan.py")
audit = read_json(AUDIT_PATH)
show_table(["Comprobación", "Resultado observado"], [
    ["Filas / intentos originales", audit["raw"]["rows"]],
    ["Pedidos distintos", audit["raw"]["distinct_orders"]],
    ["Pedidos con reintentos", audit["raw"]["orders_with_repeated_attempts"]],
    ["Eventos arrive_time = 0", audit["raw"]["zero_timestamps"]["arrive_time"]],
    ["Celdas vacías", sum(audit["raw"]["missing_counts"].values())],
    ["Pedidos elegibles", audit["cohort_proposal"]["eligible_orders"]],
    ["Decisión", audit["decision"]],
])
show_table(["Primera causa de exclusión de pedidos", "Cantidad"],
           audit["cohort_proposal"]["excluded_orders_by_first_reason"].items())
assert audit["raw"]["distinct_orders"] == (
    audit["cohort_proposal"]["eligible_orders"]
    + sum(audit["cohort_proposal"]["excluded_orders_by_first_reason"].values())
)

In [ ]:
# Resumen ya calculado por la auditoría T07; no sustituye el EDA pendiente.
show_table(["Estadística del target", "Valor observado"],
           audit["cohort_proposal"]["duration"].items())
print("Target:", audit["target"]["definition"], "—", audit["target"]["unit"])
for condition in audit["conditions"]:
    print("•", condition)

**Conclusión de T07:** la aprobación es condicional para una cohorte académica.
No demuestra precisión de un modelo. Una duración larga no se elimina por un
umbral inventado. Las coordenadas tienen unidades pendientes: no se convierten
en grados ni kilómetros. Los campos posteriores a la solicitud sirven para
verificar la etiqueta; usarlos como entrada del modelo introduciría **fuga de información**.

## T08 — Preparar con Pandas

**Objetivo:** materializar exactamente la cohorte auditada.

La función modular valida tipos e identificadores, detecta cambios entre intentos,
filtra pedidos elegibles y deriva duración, hora y día UTC. Devuelve una tabla
nueva y conserva trazabilidad. No aprende imputaciones, escalados ni modelos.

La siguiente celda muestra el código real de Pandas usado por la receta; evita
mantener una copia divergente dentro del notebook.

In [ ]:
_ = run_project("python", "-c", "import inspect; from delivery_ml.preparation import prepare_orders; print(inspect.getsource(prepare_orders))")

In [ ]:
_ = run_project("python", "ml/data/prepare_meituan.py")
prepared = read_json(PROCESSED_DIR / "manifest.json")
assert prepared["rows"] == audit["cohort_proposal"]["eligible_orders"]
assert prepared["source_sha256"] == audit["source"]["sha256"]
assert prepared["audit_sha256"] == sha256_file(PROJECT_DIR / AUDIT_PATH)
assert prepared["sha256"] == sha256_file(PROJECT_DIR / PROCESSED_DIR / "orders.csv")
show_table(["Evidencia de preparación", "Valor"], [
    ["Filas", prepared["rows"]], ["Columnas", len(prepared["columns"])],
    ["Bytes", prepared["bytes"]], ["SHA-256", prepared["sha256"]],
    ["Pandas", prepared["pandas_version"]], ["Zona horaria", prepared["timezone"]],
])
show_table(["Papel", "Columnas / tratamiento"], [
    ["Candidatas de contexto", ", ".join(prepared["candidate_features"])],
    ["Etiqueta observada", prepared["target"]],
    ["Trazabilidad; excluir de predictores", "order_id, source_waybill_id, platform_order_time, arrive_time"],
    ["Geografía", prepared["geography"]],
])
print("Entrenamiento:", prepared["training"])

## Comprobar que se puede repetir

La receta verifica la copia preparada sin sobrescribirla cuando coincide con
fuente, auditoría e implementación. Las pruebas unitarias utilizan casos
controlados exclusivamente para verificar reglas; esos casos nunca se agregan
al dataset real ni se usan para entrenar.

In [ ]:
previous_hash = prepared["sha256"]
_ = run_project("python", "ml/data/prepare_meituan.py")
assert sha256_file(PROJECT_DIR / PROCESSED_DIR / "orders.csv") == previous_hash
print("Salida sin cambios; hash verificado.")
_ = run_project("pytest", "-q")

## T09 — EDA y visualizaciones

**Objetivo:** estudiar distribución, anomalías, relaciones y sesgos de la cohorte.
El script verifica el hash T08, genera agregados y guarda cuatro figuras. No modifica
el CSV ni decide features o umbrales de un modelo.

El criterio estadístico Q3 + 1.5×IQR señala una cola para revisión; no demuestra
que esos pedidos sean errores ni establece una duración máxima comercial.
[Referencia NIST](https://www.itl.nist.gov/div898/handbook/eda/section3/boxplot.htm).

Las asociaciones con hora y día se presentan por grupos, porque el contexto es
cíclico. La matriz usa Spearman (Pearson sobre rangos con empates promediados)
solo para el target y las coordenadas originales. Se excluyen identificadores y
eventos posteriores; no se infieren kilómetros ni utilidad predictiva.


In [ ]:
_ = run_project("python", "ml/data/explore_meituan.py")
EDA_DIR = Path("data/reports/meituan") / REVISION / "t09"
eda = read_json(EDA_DIR / "t09_eda.json")
assert eda["provenance"]["processed_sha256"] == prepared["sha256"]
assert eda["rows"] == prepared["rows"]
assert sha256_file(PROJECT_DIR / PROCESSED_DIR / "orders.csv") == prepared["sha256"]
show_table(["Calidad", "Resultado"], [
    ["Celdas ausentes", sum(eda["quality"]["missing_by_column"].values())],
    ["Pedidos duplicados", eda["quality"]["duplicate_orders"]],
    ["Target distinto de eventos", eda["quality"]["target_mismatch"]],
    ["Contexto distinto del timestamp UTC", eda["quality"]["utc_context_mismatch"]],
    ["Filas descartadas por este EDA", eda["tail_diagnostics"]["removed_rows"]],
])


In [ ]:
from IPython.display import Image, Markdown

for name in eda["figures"]:
    display(Image(filename=str(PROJECT_DIR / EDA_DIR / name)))
display(Markdown("### Hallazgos calculados\n\n" + "\n\n".join(eda["findings"])))


**Cómo interpretar los gráficos:** el primer histograma conserva toda la cola;
el segundo amplía la región hasta P99 y especifica cuántos pedidos quedan fuera
solo de esa vista. P10–P90 describe dispersión observada, no incertidumbre de la
mediana. Comparar horas/días exige mirar también sus conteos y la cobertura
parcial de los bordes. Los patrones son descriptivos y no prueban causalidad.

El período es corto, procede de China y selecciona entregas completadas e
inmediatas. No valida predicciones locales ni por vehículo. T13 definirá una
partición sin fuga; este EDA no entrena ni usa un conjunto de prueba para tuning.

## T10–T11 — Dask y PySpark local · Pendiente

Repetir transformaciones acordadas usando la misma fuente y criterios.
Mostrar código, comprobar equivalencia con Pandas y registrar versiones/entorno.
No instalar estos motores hasta trabajar sus tareas.

## T12 — Comparación · Pendiente

Medir operaciones equivalentes sobre los mismos datos y equipo, documentando
arranque, lectura y procesamiento. Reportar tiempos reales; no suponer que un
motor distribuido será más rápido para este volumen.

## Para la exposición

1. ¿Por qué una fila original no equivale a un pedido de entrenamiento?
2. ¿Por qué un timestamp cero importa aunque no haya nulos?
3. ¿Qué diferencia hay entre etiqueta, predictor y columna de trazabilidad?
4. ¿Qué demuestra el hash y qué demuestra la auditoría?
5. ¿Por qué estos resultados no validan un servicio en El Salvador?

La guía personal externa explica las decisiones del proceso. Este notebook
permite ejecutarlas y mostrar sus evidencias. El entrenamiento T13–T19 tendrá
su propio cuaderno cuando se implementen esas tareas.